# The Haldane gap

In this tutorial we will calculate the Haldane gap (the energy gap in the $S = 1$ Heisenberg model) in two different ways.
To follow the tutorial you need the following packages:

In [ ]:
using MPSKit, TensorKit, Plots, Polynomials
# Model constructors for the MPSKit documentation and examples.
# Adapted from MPSKitModels.jl v0.4.7, src/models/{hamiltonians,transfermatrices}.jl:
# https://github.com/QuantumKitHub/MPSKitModels.jl/tree/v0.4.7/src/models
# Operators come from TensorKitTensors; chain assembly uses MPSKit directly.
#
# MIT License
# Copyright (c) 2021 Maarten Van Damme
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

module ExampleModels

using MPSKit, TensorKit
using TensorKitTensors: SpinOperators, BosonOperators, HubbardOperators

export chain_hamiltonian, transverse_field_ising, heisenberg_XXX
export hubbard_model, bose_hubbard_model, hard_hexagon

# L specifies an open finite chain; otherwise unitcell specifies an infinite chain.
function chain_hamiltonian(twosite, onesite = nothing; L = nothing, unitcell::Integer = 1)
    nsites = isnothing(L) ? unitcell : L
    nsites isa Integer && nsites > 0 || throw(ArgumentError("chain length must be a positive integer"))
    nbonds = isnothing(L) ? nsites : nsites - 1
    terms = Pair[(i, i + 1) => twosite for i in 1:nbonds]
    if !isnothing(onesite)
        append!(terms, [i => onesite for i in 1:nsites])
    end
    spaces = fill(space(twosite, 1), nsites)
    constructor = isnothing(L) ? InfiniteMPOHamiltonian : FiniteMPOHamiltonian
    return constructor(spaces, terms)
end

# H = -J sum(σz_i σz_{i+1} + g σx_i).
function transverse_field_ising(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, g = 1.0, kwargs...
    )
    ZZ = 4 * SpinOperators.S_z_S_z(T, symmetry; spin = 1 // 2)
    X = SpinOperators.σˣ(T, symmetry)
    return chain_hamiltonian(-J * ZZ, -J * g * X; kwargs...)
end
transverse_field_ising(symmetry::Type{<:Sector}; kwargs...) =
    transverse_field_ising(ComplexF64, symmetry; kwargs...)

# H = J sum(S_i ⋅ S_{i+1}); preserve the examples' spin-1 default.
function heisenberg_XXX(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        J = 1.0, spin = 1, kwargs...
    )
    term = J * SpinOperators.S_exchange(T, symmetry; spin)
    return chain_hamiltonian(term; kwargs...)
end
heisenberg_XXX(symmetry::Type{<:Sector}; kwargs...) =
    heisenberg_XXX(ComplexF64, symmetry; kwargs...)

# H = -t sum(e⁺_i e⁻_{i+1} + h.c.) + U sum(n↑ n↓) - mu sum(n).
function hubbard_model(
        T::Type{<:Number} = ComplexF64, particle_symmetry::Type{<:Sector} = Trivial,
        spin_symmetry::Type{<:Sector} = Trivial;
        t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    # e_hopping includes the fermionic reordering sign and is Hermitian.
    hopping = HubbardOperators.e_hopping(T, particle_symmetry, spin_symmetry)
    interaction = HubbardOperators.ud_num(T, particle_symmetry, spin_symmetry)
    N = HubbardOperators.e_num(T, particle_symmetry, spin_symmetry)
    return chain_hamiltonian(-t * hopping, U * interaction - mu * N; kwargs...)
end

# H = -t sum(b⁺_i b⁻_{i+1} + h.c.) + U/2 sum(n(n-1)) - mu sum(n).
function bose_hubbard_model(
        T::Type{<:Number} = ComplexF64, symmetry::Type{<:Sector} = Trivial;
        cutoff::Integer = 5, t = 1.0, U = 1.0, mu = 0.0, kwargs...
    )
    hopping = BosonOperators.b_plus_b_min(T, symmetry; cutoff) +
        BosonOperators.b_min_b_plus(T, symmetry; cutoff)
    N = BosonOperators.b_num(T, symmetry; cutoff)
    interaction = N * (N - id(domain(N)))
    return chain_hamiltonian(-t * hopping, U / 2 * interaction - mu * N; kwargs...)
end

function hard_hexagon(T::Type{<:Number} = ComplexF64)
    P = Vect[FibonacciAnyon](:τ => 1)
    O = ones(T, P ⊗ P ← P ⊗ P)
    block(O, FibonacciAnyon(:I)) .*= 0
    return InfiniteMPO([O])
end

end

using .ExampleModels
using TensorKitTensors.SpinOperators: S_exchange

The Heisenberg model is defined by the following Hamiltonian:

$$
H = -J∑_{⟨i,j⟩} (X_i X_j + Y_i Y_j + Z_i Z_j)
$$

This Hamiltonian has an SU(2) symmetry, which we can enforce by using SU(2)-symmetric tensors:

In [ ]:
symmetry = SU2Irrep
spin = 1
J = 1

## Finite size extrapolation

We can start the analysis using finite-size methods.
The ground state of this model can be approximated using finite MPS through the use of DMRG.

The typical way to find excited states is to minimize the energy while adding an error term
$$λ \left|gs\right> \left< gs\right|$$
Here we will instead use the [quasiparticle ansatz](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.111.080401).

In Steven White's original DMRG paper it was remarked that the $S = 1$ excitations correspond to edge states, and that one should define the Haldane gap as the difference in energy between the $S = 2$ and $S = 1$ states.
This can be done as follows.

In [ ]:
L = 11
H = heisenberg_XXX(symmetry; L, J, spin)

physical_space = SU2Space(1 => 1)
virtual_space = SU2Space(0 => 12, 1 => 12, 2 => 5, 3 => 3)
ψ₀ = FiniteMPS(L, physical_space, virtual_space)
ψ, envs, info = find_groundstate(ψ₀, H, DMRG(; verbosity = 0))
E₀ = real(expectation_value(ψ, H))
En_1, st_1 = excitations(H, QuasiparticleAnsatz(), ψ, envs; sector = SU2Irrep(1))
En_2, st_2 = excitations(H, QuasiparticleAnsatz(), ψ, envs; sector = SU2Irrep(2))
ΔE_finite = real(En_2[1] - En_1[1])

We can go even further and doublecheck the claim that $S = 1$ is an edge excitation, by plotting the energy density.

In [ ]:
p_density = plot(; xaxis = "position", yaxis = "energy density")
excited_1 = convert(FiniteMPS, st_1[1])
excited_2 = convert(FiniteMPS, st_2[1])
SS = -S_exchange(ComplexF64, SU2Irrep; spin = 1)
e₀ = [real(expectation_value(ψ, (i, i + 1) => SS)) for i in 1:(L - 1)]
e₁ = [real(expectation_value(excited_1, (i, i + 1) => SS)) for i in 1:(L - 1)]
e₂ = [real(expectation_value(excited_2, (i, i + 1) => SS)) for i in 1:(L - 1)]
plot!(p_density, e₀; label = "S = 0")
plot!(p_density, e₁; label = "S = 1")
plot!(p_density, e₂; label = "S = 2")

Finally, we can obtain a value for the Haldane gap by extrapolating our results for different system sizes.

In [ ]:
Ls = 12:4:30
ΔEs = map(Ls) do L
    @info "computing L = $L"
    ψ₀ = FiniteMPS(L, physical_space, virtual_space)
    H = heisenberg_XXX(symmetry; L, J, spin)
    ψ, envs, info = find_groundstate(ψ₀, H, DMRG(; verbosity = 0))
    En_1, st_1 = excitations(H, QuasiparticleAnsatz(), ψ, envs; sector = SU2Irrep(1))
    En_2, st_2 = excitations(H, QuasiparticleAnsatz(), ψ, envs; sector = SU2Irrep(2))
    return real(En_2[1] - En_1[1])
end

f = fit(Ls .^ (-2), ΔEs, 1)
ΔE_extrapolated = f.coeffs[1]
p_size_extrapolation = plot(; xaxis = "L^(-2)", yaxis = "ΔE", xlims = (0, 0.015))
plot!(p_size_extrapolation, Ls .^ (-2), ΔEs; seriestype = :scatter, label = "numerical")
plot!(p_size_extrapolation, x -> f(x); label = "fit")

## Thermodynamic limit

A much nicer way of obtaining the Haldane gap is by working directly in the thermodynamic limit.
As was already hinted at by the edge modes, this model is in a non-trivial SPT phase.
Thus, care must be taken when selecting the symmetry sectors.
The ground state has half-integer edge modes, thus the virtual spaces must also all carry half-integer charges.

In contrast with the finite size case, we now should specify a momentum label to the excitations.
This way, it is possible to scan the dispersion relation over the entire momentum space.

In [ ]:
H = heisenberg_XXX(symmetry; J, spin)
virtual_space_inf = Rep[SU₂](1 // 2 => 16, 3 // 2 => 16, 5 // 2 => 8, 7 // 2 => 4)
ψ₀_inf = InfiniteMPS([physical_space], [virtual_space_inf])
ψ_inf, envs_inf, info_inf = find_groundstate(ψ₀_inf, H; verbosity = 0)

kspace = range(0, π, 16)
Es, _ = excitations(H, QuasiparticleAnsatz(), kspace, ψ_inf, envs_inf; sector = SU2Irrep(1))

ΔE, idx = findmin(real.(Es))
println("minimum @k = $(kspace[idx]):\t ΔE = $(ΔE)")
plot(kspace, real.(Es); xaxis = "momentum", yaxis = "ΔE", label = "S = 1")

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*